In [ ]:
import numpy as np, pandas as pd, scanpy as sc, matplotlib.pyplot as plt, os
from scipy.stats import hypergeom
import celloracle as co, glob, pickle
from functools import reduce
from tqdm import tqdm
import itertools, math, random
import networkx as nx

# visualization settings required to see plots in jupyter notebook
%config InlineBackend.figure_format = 'retina'
%matplotlib inline
plt.rcParams['figure.figsize'] = [6, 4.5]
plt.rcParams["savefig.dpi"] = 300

wd = '/ocean/projects/cis240075p/skeshari/igvf/bcell2/male_donor/'
out_path = os.path.join(wd, 'out_data', 'lf_enrich')
os.makedirs(f"{out_path}/figures", exist_ok=True)
os.makedirs(f"{out_path}/out_files", exist_ok=True)
sc.settings.figdir = f"{out_path}/figures"
random.seed(42)

In [1]:
from utils import *

/ocean/projects/cis240075p/asachan/.conda/envs/celloracle_env/lib/python3.10/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


In [ ]:
#### Create input dictionary for the experiments
input_dict = {
    'experiment': ['PRDM1_KO', 'IRF4_KO', 'GC_PB', 'PB_ABC', 'GC_ABC'],
    'slide_starting_genes': [4472, 4500, 4725, 3420, 4603],
    'clusters_of_interest': [['3','7'], ['3','7'], ['7','3'], ['1','7'], ['1','3']],
    'order_fr_clust': [[2], [2], [2], [2], [2]],
    'order_fr_tfcomb': [[1], [1], [2], [2], [2]],
    'weight': ['strength', 'strength', 'strength', 'strength', 'strength'],
}
input_df = pd.DataFrame(input_dict)

#### Assign the input parameters
i=1
experiment = input_df['experiment'][i]
slide_starting_genes = input_df['slide_starting_genes'][i]
clusters_of_interest = input_df['clusters_of_interest'][i]
order_fr_clust = input_df['order_fr_clust'][i]
order_fr_tfcomb = input_df['order_fr_tfcomb'][i]
weight = input_df['weight'][i]

# ------------------------------------------------------------
# Reading the data
# ------------------------------------------------------------
#Read the GRN data and slide features
GRN_wd = '/ocean/projects/cis240075p/skeshari/igvf/bcell1/male_donor'
GRN_links_after_fit, GRN_network_scores, GRN_TFs = fetch_GRN_data(GRN_wd)
slide_features = read_slide_data(experiment, wd)
cluster_fusions = []
for ord_clus in order_fr_clust:
    cluster_fusions += list(itertools.combinations(clusters_of_interest, ord_clus))

In [ ]:
create_df_pickle = True
for cluster_fusion in cluster_fusions:
    combined_links, threshold = create_combined_links_for_cluster_fusion(cluster_fusion, GRN_links_after_fit, quantile=0.90)
    grn, edges_df = filter_combined_links_and_build_grn(combined_links, threshold)

    fig = edges_df.groupby(['strength', 'key']).size().unstack().plot(kind='bar', stacked=True)
    plt.xlabel('Strength')
    plt.ylabel('Count')
    plt.title('Distribution of Key and Strength')
    plt.savefig(f"{out_path}/figures/combined_links_key_strength_{cluster_fusion}{experiment}.pdf")
    plt.close()
    # ------------------------------------------------------------
    # Creating the TF lists
    # ------------------------------------------------------------
    # Step1: Filter SLIDE data wrt GRN, since SLIDE and CO gene sets are not same
    slide_features = slide_features.intersection(set(grn.nodes)) 
    slide_features_neighbors = []
    for gene in slide_features:
        slide_features_neighbors += list(grn.predecessors(gene))
    slide_tot_TF = (slide_features.union(set(slide_features_neighbors))).intersection(GRN_TFs)

    # # Step2: Creating the list of network and random TFs only for analysis for order of TF combinations >1
    # # Read network scores
    # combined_network_scores = filter_network_score_data(cluster_fusion, GRN_network_scores)
    # combined_network_scores = combined_network_scores[combined_network_scores.index.isin(list(grn.nodes))] # Since SLIDE and CO gene sets are not same
    # combined_network_scores = combined_network_scores[combined_network_scores.index.isin(GRN_TFs)] # Since I want to pick size matched set of TFs only
    # #### Choosing size matched set of TFs from network and random
    # net_match_TF = set(combined_network_scores.index[:len(slide_tot_TF)])
    # # net_rnd_TF = set(random.sample(set(GRN_TFs).intersection(set(grn.nodes)), len(slide_tot_TF)))
    # net_rnd_TF = set(random.sample(set(GRN_TFs), len(slide_tot_TF)))

    # # write the TFs to file as separate csvs
    # pd.DataFrame(slide_tot_TF).to_csv(f"{out_path}/out_files/SLIDE_LF_{cluster_fusion}_{experiment}.csv", index=False)
    # pd.DataFrame(net_match_TF).to_csv(f"{out_path}/out_files/Net_match_{cluster_fusion}_{experiment}.csv", index=False)
    # pd.DataFrame(net_rnd_TF).to_csv(f"{out_path}/out_files/Net_rnd_{cluster_fusion}_{experiment}.csv", index=False)
    # # Create a Venn diagram for the three sets
    # from matplotlib_venn import venn3
    # venn = venn3([slide_tot_TF, set(net_match_TF), set(net_rnd_TF)], ('Slide', 'Net', 'Random'))
    # plt.show()


    # ------------------------------------------------------------
    # Finally doing enrichments
    # ------------------------------------------------------------
    for ord_tf in order_fr_tfcomb:
        cc_dict = {} # Initialize the dictionary to store enrichment results for each cluster fusion and order of TF combinations
        # Step1: Deciding TF combinations for which the enrichment needs to be done
        if ord_tf == 1:
            cases = [(slide_tot_TF, 'slide')]
        elif ord_tf == 2:
            # common_TFs = slide_tot_TF.intersection(net_match_TF.intersection(net_rnd_TF)) 
            # slide_tot_TF = slide_tot_TF- common_TFs
            # net_match_TF = net_match_TF - common_TFs
            # net_rnd_TF = net_rnd_TF - common_TFs
            cases = [(slide_tot_TF, 'slide')] #[(slide_tot_TF, 'slide'),(net_match_TF, 'net'),(net_rnd_TF, 'rnd')]
        # Step2: Performing enrichment analysis
        for TFs, case in cases:
            print(f"Running enrichment for {experiment}, {cluster_fusion}, {ord_tf} TFs, case = {case}")
            cc_dict = get_SLIDE_GRN_enrichment(edges_df,cc_dict,cluster_fusion,ord_tf,slide_features,slide_starting_genes,TFs,case)
        # ------------------------------------------------------------
        # Optionally generating the dataframe to be saved
        # ------------------------------------------------------------
        if create_df_pickle == True:
            suffix = f"_{experiment}"
            print(f"Creating enrichment dataframe for {experiment}, {cluster_fusion}, {ord_tf} TFs, all cases")
            enrichment_df= create_enrichment_df(cc_dict, cluster_fusion, ord_tf, filter = None, suffix=suffix)
            print(enrichment_df['case'].value_counts())
            # dumping the dictionary to pickle file
            with open(f"{out_path}/out_files/SLIDE_LF_enrichment/cc_dict_{ord_tf}_TFs_{cluster_fusion}{suffix}.pickle", 'wb') as f:
                pickle.dump(cc_dict, f)



# Plotting enrichment scores per cluster